# Connect4

-----general introduction to the work----------

In [1]:
import random
import math 
import csv
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from __future__ import print_function 


## Game Logic

In [2]:
ROWS = 6
COLS = 7

def create_board():
  
    return [[0] * COLS for _ in range(ROWS)]

def print_board(board):
    
    for row in board:
        print(" ".join("X" if cell == 1 else "O" if cell == 2 else "-" for cell in row))

def is_valid_move(board, col):
    
    return board[0][col] == 0

def get_next_open_row(board, col):
    
    for row in range(ROWS - 1, -1, -1):  
        if board[row][col] == 0:
            return row
    print("[ERROR] No row available")
    return None  

def drop_piece(board, row, col, piece):
    
    board[row][col] = piece

def check_winner(board, piece):
    
    
    for r in range(ROWS):
        for c in range(COLS - 3):
            if all(board[r][c + i] == piece for i in range(4)):
                return True

    
    for r in range(ROWS - 3):
        for c in range(COLS):
            if all(board[r + i][c] == piece for i in range(4)):
                return True

    
    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            if all(board[r + i][c + i] == piece for i in range(4)):
                return True

    
    for r in range(3, ROWS):
        for c in range(COLS - 3):
            if all(board[r - i][c + i] == piece for i in range(4)):
                return True

    return False

def get_valid_moves(board):

    return [column for column in range(COLS) if is_valid_move(board, column)]

def board_to_string(board):
    return ''.join([str(cell) for row in board for cell in row])

 ## Random Algorithm


In [3]:
def get_random_move(**kwargs):
    board = kwargs.get("board")
    
    return random.choice(get_valid_moves(board))

## Monte Carlo Tree Search Algorithm

In [4]:
class MCTSNode:
    def __init__(self, board, player, parent=None, move=None):
        self.board = [row[:] for row in board]  
        self.player = player          
        self.parent = parent              
        self.move = move                 
        self.children = []                
        self.visits = 0                  
        self.wins = 0                  
        self.untried_moves = get_valid_moves(board)  

    def ucb1(self, total_simulations, c=1.41):
        if self.visits == 0:
            return float('inf')  
        win_rate = self.wins / self.visits
        return win_rate + c * math.sqrt(math.log(total_simulations) / self.visits)

    def is_fully_expanded(self):
        return len(self.untried_moves) == 0 or check_winner(self.board,1) or check_winner(self.board,2)

def MCTS(**kwargs):
    root = kwargs.get("root")
    iterations = kwargs.get("iterations", 500)
    bestchild = kwargs.get("bestchild", bestchild_default)
    backprograming = kwargs.get("backprograming", backprograming_default)
    c = kwargs.get("c", 1.41)

    for _ in range(iterations):
        node = selection(root,c=c)
        if not node.is_fully_expanded():
            child = expansion(node)
            result = simulation(child)
            backprograming(child, result)
        else:
            result = node.player if check_winner(node.board,  node.player) else 0
            backprograming(node, result)

    chosen = bestchild(root)
    if chosen is None:
        print("[MCTS DEBUG] No child chosen — fallback to random move.")
        return random.choice(get_valid_moves(root.board))

    return chosen.move

def selection(node: MCTSNode, c=1.41):
    while True:
    
        if not node.is_fully_expanded():
            return node

        if not node.children:
            return node

        node = max(node.children, key=lambda child: child.ucb1(node.visits,c))

def expansion(node: MCTSNode) ->MCTSNode:
    
    move=random.choice(node.untried_moves)
    node.untried_moves.remove(move)
    
    row=get_next_open_row(node.board,move)
    new_board=[row[:] for row in node.board]
    
    drop_piece(new_board,row,move,3-node.player)


    expanded_node=MCTSNode(board=new_board,player=3-node.player,parent=node,move=move)

    node.children.append(expanded_node)

    return expanded_node

def get_smart_move(board, player):
  
    for col in get_valid_moves(board):
        row = get_next_open_row(board, col)
        temp_board = [r[:] for r in board]
        drop_piece(temp_board, row, col, player)
        if check_winner(temp_board, player):
            return col

   
    opponent = 3 - player
    for col in get_valid_moves(board):
        row = get_next_open_row(board, col)
        temp_board = [r[:] for r in board]
        drop_piece(temp_board, row, col, opponent)
        if check_winner(temp_board, opponent):
            return col

   
    return random.choice(get_valid_moves(board))

def simulation(node: MCTSNode) ->int:
    board=[row[:] for row in node.board]
    player= 3 - node.player
    while get_valid_moves(board):
    
        move=get_smart_move(board=board,player=player)
        row=get_next_open_row(board,move)

        drop_piece(board,row,move,player)
        
        if check_winner(board,player):
            return player
        player=3-player
    return 0

def backprograming_default(node : MCTSNode,result):
    
    while node is not None:
        node.visits+=1

        if result== node.player: 
            
            node.wins+=1

        elif result==0:
            node.wins+=0.5
        node=node.parent

def backprograming_greddy(node : MCTSNode,result):

    while node is not None:
        node.visits+=1

        if result== node.player: 
            
            node.wins+=1

        elif result==0:
            node.wins+=0.15
        node=node.parent

def bestchild_default(node: MCTSNode):
    if not node.children:
        print("[BESTCHILD DEBUG] No children in node.")
        return None

    visited_children = [child for child in node.children if child.visits > 0]
    if not visited_children:
        print("[BESTCHILD DEBUG] All children have 0 visits.")
        return None

    best = max(visited_children, key=lambda child: child.wins / child.visits)

    return best

def bestchild_higherVisits(node : MCTSNode):
    if not node.children:
        print("[BESTCHILD DEBUG] No children in node.")
        return None

    visited_children = [child for child in node.children if child.visits > 0]
    if not visited_children:
        print("[BESTCHILD DEBUG] All children have 0 visits.")
        return None

    best = max(visited_children, key=lambda child: child.visits)

    return best

def update_root(root,board,col,turn):   

    matching_child = next((child for child in root.children if child.move == col), None)
    if matching_child:
        root = matching_child
        root.parent = None
    else:
       
        root = MCTSNode(board=board, player= turn)
    return root 


### Save moves data

In [5]:
def save_to_csv(data, filename="mcts_moves.csv"):
    fieldnames = [f"s{i}" for i in range(42)] + ["move"]

    write_header = not os.path.exists(filename)

    with open(filename, mode='a', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)

        if write_header:
            writer.writeheader()

        for entry in data:
            state_str = entry['state'] 
            move = entry['move']

            row = {f"s{i}": int(state_str[i]) for i in range(42)}
            row["move"] = move

            writer.writerow(row)

### See data

In [6]:
from collections import Counter

def analyse_game_data():
    df=pd.read_csv("mcts_moves.csv")

    board_cols = [f"s{i}" for i in range(42)]
    boards = df[board_cols]
    
    states = boards.apply(lambda row: tuple(row), axis=1)
    
    state_counts = Counter(states)
    
    print("Top 100 estados mais frequentes:")
    for state, count in state_counts.most_common(100):
        print(f"{state} → {count} vezes")
    
    print(f"\nTotal de estados únicos: {len(state_counts)}")

In [7]:
analyse_game_data()

Top 100 estados mais frequentes:
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0) → 1062 vezes
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0) → 496 vezes
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0) → 196 vezes
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0) → 150 vezes
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0) → 149 vezes
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0) → 128 vezes
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0) → 110

### Get id3 move

In [8]:
def ID3(**kwargs):
    board = kwargs.get("board")

    with open('connect4.pkl' , "rb") as f:
        tree = pickle.load(f)
    
    state=board_to_state(board)

    move= tree.predict(pd.DataFrame([state]))[0]

   
    
    if move not in get_valid_moves(board):
        print("[ID3] Move out of the board")
        
        move=random.choice(get_valid_moves(board))

        print(f'[ID3] random move {move}')
    
    return move
    
def board_to_state(board):
    state={}
    index= 0
    for row in board:
        for cell in row:
            state[f"s{index}"] = cell
            index+=1
    return state

## Player vs Player/Ai

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def play_game(mode="pvc", ai=None):
    board = create_board()
    game_over = False
    turn = 1
    root = None


    last_move = None
    last_symbol = None
    last_player = None

    if ai == MCTS:
        root = MCTSNode(board, 1)

    
        
    output = widgets.Output()
    display(output)

    def render():
        with output:
            clear_output(wait=True)
            print_board(board)

           
            if last_move is not None and last_symbol is not None and last_player is not None:
                if mode == "pvc" and last_player == 2:
                    jogador = "Computador"
                else:
                    jogador = "Jogador"
                print(f"Última jogada: {jogador} ({last_symbol}) jogou na coluna {last_move}")
            
           
 
            proximo_simbolo = 'X' if turn == 1 else 'O'
            proximo_jogador = "Jogador" if (mode == "pvp" or turn == 1) else "Computador"
            print(f"Próximo a jogar: {proximo_jogador} ({proximo_simbolo})")

    def on_button_click(b):
        nonlocal turn, board, root, game_over
        nonlocal last_move, last_symbol, last_player

        if game_over:
            return

        col = int(b.description)

        if col not in get_valid_moves(board):
            with output:
                clear_output(wait=True)
                print_board(board)
                print("Jogada inválida! Tenta outra.")
            return

        row = get_next_open_row(board, col)
        drop_piece(board, row, col, turn)


        last_move = col
        last_symbol = 'X' if turn == 1 else 'O'
        last_player = turn

        if root:
            root = update_root(root, board, col, turn)

        turn = 3 - turn 
        render()

        if check_winner(board, 3- turn):
            with output:
                print(f"Jogador {3-turn} ({last_symbol}) venceu!")
            game_over = True
            return
        elif not get_valid_moves(board):
            with output:
                print("Empate!")
            game_over = True
            return

        if mode == "pvc" and turn == 2 and ai:
            col = ai(root=root, board=board)
            row = get_next_open_row(board, col)
            drop_piece(board, row, col, turn)
            last_move = col
            last_symbol = 'O'
            last_player = 2
            if root:
                root = update_root(root, board, col, turn)

            turn = 3 - turn
            render()
            if check_winner(board, 3- turn):
                with output:
                    print("Computador (O) venceu!")
                game_over = True
                return
            elif not get_valid_moves(board):
                with output:
                    print("Empate!")
                game_over = True
                return
            
    buttons = [widgets.Button(description=str(i)) for i in range(COLS)]
    for btn in buttons:
        btn.on_click(on_button_click)

    display(widgets.HBox(buttons))
    render()



In [ ]:
play_game('pvc' , get_random_move)

In [ ]:
play_game(mode= "pvp")

In [ ]:
play_game(mode="pvc", ai= MCTS)

## C v C

In [10]:
def simulate_game(ai1,ai2,silent=True, save=False):
    board=create_board()
    root1=None
    root2=None
    turn=1
    if callable(ai1):
        try:
            root1 = MCTSNode(board, 2)  
        except:
            pass
    if callable(ai2):
        try:
            root2 = MCTSNode(board, 1)
        except:
            pass

    
    game_data=[]
    
    while True:
       
        winner = 3 - turn if check_winner(board, 3 - turn) else 0

        if winner != 0 or not get_valid_moves(board):
            
            if winner:
                print(f"Player {winner} ({'X' if winner == 1 else 'O'}) wins!")
            else:
                print("It's a draw!")

            if save:
                save_to_csv(game_data) 
            
            return winner

        if turn==1:
            col=ai1(board=board,root=root1)
            if getattr(ai1, "_is_mcts", False):
                
                game_data.append({
                'state': board_to_string(board),
                'move': col,
                })
        else:
            col=ai2(board=board,root=root2)
            if getattr(ai2, "_is_mcts", False):
                game_data.append({
                'state': board_to_string(board),
                'move': col,   
                }) 

        row=get_next_open_row(board,col)
        drop_piece(board,row,col,turn)

        if not silent:
            print_board(board)
            print(f"Computer {turn} chooses column {col}")

        if root1:
            root1=update_root(root1,board,col,turn)  
                  
        if root2:
            root2=update_root(root2,board,col,turn) 
 
        turn= 3-turn

In [ ]:
simulate_game(MCTS,get_random_move,silent=False)

## Benchmark algorithms

In [11]:
def select_mcts_parameters(
    c=1.41,
    iterations=500,
    bestchild_name="bestchild_default",
    backprograming_name="backprograming_default"
):

    bestchild_options = {
        "bestchild_default":  bestchild_default,
        "bestchild_higherVisits": bestchild_higherVisits
    }

    backprograming_options = {
        "backprograming_default": backprograming_default,
        "backprograming_greddy": backprograming_greddy
    }

   
    if bestchild_name not in bestchild_options:
        raise ValueError(f"Função bestchild '{bestchild_name}' não encontrada.")
    if backprograming_name not in backprograming_options:
        raise ValueError(f"Função backprograming '{backprograming_name}' não encontrada.")

    return {
        "c": c,
        "iterations": iterations,
        "bestchild": bestchild_options[bestchild_name],
        "backprograming": backprograming_options[backprograming_name]
    }

def benchmark(strategy1, strategy2, games=20,silent=True, name1="AI1" , name2="AI2", save=False):
    ai1_wins = 0
    draws = 0

    for i in range(games):
        print(f"Simulating game {i+1} ...")
        winner = simulate_game( ai1=strategy1, ai2=strategy2,silent=silent,save=save)
        if winner == 1:
            ai1_wins += 1
        elif winner == 0:
            draws += 1
    print(f"{name1}: {ai1_wins}, Draws: {draws}, {name2}: {games - ai1_wins - draws}")
    
def benchmark_menu_jupyter():
    output = widgets.Output()
    
    
    ai_options = ["random", "mcts" , "id3"]
    ai1_dropdown = widgets.Dropdown(options=ai_options, description="AI 1:")
    ai2_dropdown = widgets.Dropdown(options=ai_options, description="AI 2:")

    
    games_slider = widgets.IntSlider(value=10, min=1, max=1000, step=1, description="Games:")

    
    show_board = widgets.Checkbox(value=False, description="Show board")

    save_moves= widgets.Checkbox(value=False, description='Save moves ')
    
    start_button = widgets.Button(description="Start Benchmark", button_style="success")

    
    mcts1_box = widgets.VBox()
    mcts2_box = widgets.VBox()

    def create_mcts_config(player_label="MCTS"):
        use_default = widgets.Checkbox(value=True, description=f"{player_label}: use default")

        
        
        bestchild = widgets.Dropdown(
            options=["bestchild_default", "bestchild_higherVisits"], 
            description="Bestchild:"
        )
        backprog = widgets.Dropdown(
            options=["backprograming_default", "backprograming_greddy"], 
            description="Backprog:"
        )
        c = widgets.FloatText(value=1.41, description="c:")
        iterations = widgets.IntText(value=500, description="Iterations:")

        advanced_box = widgets.VBox([bestchild, backprog, c, iterations])
        advanced_box.layout.display = 'none'  

        def toggle_advanced_fields(change=None):
            advanced_box.layout.display = 'none' if use_default.value else 'block'

        use_default.observe(toggle_advanced_fields, names='value')

        box = widgets.VBox([use_default, advanced_box])

        def get_params():
            if use_default.value:
                return select_mcts_parameters()
            else:
                return select_mcts_parameters(
                    c=c.value,
                    iterations=iterations.value,
                    bestchild_name=bestchild.value,
                    backprograming_name=backprog.value
                )

        return box, get_params


    mcts1_cfg_box, get_mcts1_params = create_mcts_config("MCTS 1")
    mcts2_cfg_box, get_mcts2_params = create_mcts_config("MCTS 2")

    mcts1_box.children = [mcts1_cfg_box]
    mcts2_box.children = [mcts2_cfg_box]

    def toggle_mcts_boxes(*args):
        mcts1_box.layout.display = 'block' if ai1_dropdown.value == "mcts" else 'none'
        mcts2_box.layout.display = 'block' if ai2_dropdown.value == "mcts" else 'none'

    ai1_dropdown.observe(toggle_mcts_boxes, names='value')
    ai2_dropdown.observe(toggle_mcts_boxes, names='value')

    toggle_mcts_boxes()  

    def on_start_clicked(b):
        with output:
            clear_output()
            print("=== Benchmark Configuration ===")
            print(f"AI 1: {ai1_dropdown.value}")
            print(f"AI 2: {ai2_dropdown.value}")
            print(f"Games: {games_slider.value}")
            print(f"Show board: {'Yes' if show_board.value else 'No'}")
            
            
            mcts1_params = get_mcts1_params() if ai1_dropdown.value == "mcts" else {}
            mcts2_params = get_mcts2_params() if ai2_dropdown.value == "mcts" else {}

            def get_ai(name, params):
                if name == "random":
                    return get_random_move
                
                elif name == "id3":
                    return ID3

                elif name == "mcts":
                    ai_func = lambda **kwargs: MCTS(**{**kwargs, **params})
                    ai_func._is_mcts = True  
                    return ai_func
                else:
                    raise ValueError(f"Unkonwn AI : {name}")

            benchmark(
                strategy1=get_ai(ai1_dropdown.value, mcts1_params),
                strategy2=get_ai(ai2_dropdown.value, mcts2_params),
                games=games_slider.value,
                silent=not show_board.value,
                name1=ai1_dropdown.value.upper(),
                name2=ai2_dropdown.value.upper(),
                save=save_moves

            )

    start_button.on_click(on_start_clicked)

    
    display(widgets.VBox([
        widgets.HTML(value="<h3>Benchmark Menu</h3>"),
        ai1_dropdown,
        mcts1_box,
        ai2_dropdown,
        mcts2_box,
        games_slider,
        show_board,
        save_moves,
        start_button,
        output
    ]))


In [13]:
benchmark_menu_jupyter()

In [ ]:
benchmark_menu_jupyter()

In [ ]:
benchmark_menu_jupyter()

## Decision Tree Algorithm


In [ ]:
class TreeNode(object):
    def __init__(self, indices = None, children = [], entropy = 0, depth = 0):
        self.indices = indices     
        self.entropy = entropy   
        self.depth = depth       
        self.split_attribute = None 
        self.children = children 
        self.order = None      
        self.label = None      

    def set_properties(self, split_attribute, order):
        self.split_attribute = split_attribute
        self.order = order

    def set_label(self, label):
        self.label = label


def entropy(freq):
   
    freq_0 = freq[np.array(freq).nonzero()[0]]
    prob_0 = freq_0/float(freq_0.sum())
    return -np.sum(prob_0*np.log2(prob_0))

class DecisionTreeID3(object):
    def __init__(self, max_depth=10, min_samples_split=2, min_gain=1e-4):
        self.root = None
        self.max_depth = max_depth 
        self.min_samples_split = min_samples_split 
        self.Ntrain = 0
        self.min_gain = min_gain
    
    def fit(self, data, target):
        self.Ntrain = data.count().iloc[0]
        self.data = data 
        self.attributes = list(data)
        self.target = target 
        self.labels = sorted(target.unique())  
        
        indices = range(self.Ntrain)
        self.root = TreeNode(indices=indices, entropy=self._entropy(indices), depth=0)
        queue = [self.root]
        while queue:
            node = queue.pop()
            if node.depth < self.max_depth and node.entropy > self.min_gain:  
                node.children = self._split(node)
                if not node.children: 
                    self._set_label(node)
                queue += node.children
            else:
                self._set_label(node)
                
    def _entropy(self, indices):
        if len(indices) == 0: return 0
        freq = np.array(self.target.iloc[indices].value_counts())
        return entropy(freq)

    def _set_label(self, node):
        node.set_label(self.target.iloc[node.indices].mode()[0]) 
    
    def _split(self, node):
        indices = node.indices 
        best_gain = 0
        best_splits = []
        best_attribute = None
        order = None
        sub_data = self.data.iloc[indices, :]
        for i, att in enumerate(self.attributes):
            values = self.data.iloc[indices, i].unique().tolist()
            if len(values) == 1: continue 
            splits = []
            for val in values: 
                sub_indices = sub_data.index[sub_data[att] == val].tolist()
                splits.append(sub_indices)
            if min(map(len, splits)) < self.min_samples_split: continue
            HxS = 0
            for split in splits:
                HxS += len(split) * self._entropy(split) / len(indices)
            gain = node.entropy - HxS 
            if gain < self.min_gain: continue
            if gain > best_gain:
                best_gain = gain 
                best_splits = splits
                best_attribute = att
                order = values
        node.set_properties(best_attribute, order)
        child_nodes = [TreeNode(indices=split,
                     entropy=self._entropy(split), depth=node.depth + 1) for split in best_splits]
        return child_nodes

    def predict(self, new_data):
        npoints = new_data.count().iloc[0]
        labels = [None] * npoints
        for n in range(npoints):
            x = new_data.iloc[n, :]
            node = self.root
            while node.children:
                value = x[node.split_attribute]
                if value in node.order:
                    idx = node.order.index(value)
                    node = node.children[idx]
                else:
                    print('[Predict] no move found. Most comunm move')
                    
                    break
            labels[n] = node.label if node.label is not None else self.target.iloc[node.indices].mode()[0]
        return labels

    def predict_proba(self, x):
        

        node = self.root
        while node.children:
            value = x[node.split_attribute]
            if value in node.order:
                idx = node.order.index(value)
                node = node.children[idx]
            else:
                return np.ones(len(self.labels)) / len(self.labels)

        probs = np.zeros(len(self.labels))
        if node.label in self.labels:
            i = self.labels.index(node.label)
            probs[i] = 1.0
        else:
            probs[:] = 1.0 / len(self.labels)
        return probs

    def predict_probability(self, df):

        return np.array([self.predict_proba(df.iloc[i]) for i in range(len(df))])



### Prepare data

In [ ]:
def discretize(df, bins=3):
    df_disc = df.copy()
    for col in df.columns:
        df_disc[col] = pd.cut(df[col], bins, labels=False)
    return df_disc

def train_test_split(data,split=0.7,target="class"):
    data = data.sample(frac=1,random_state=42).reset_index(drop=True)


    split_size=int(len(data) *split)

    
    train_data = data.iloc[:split_size]
    test_data = data.iloc[split_size:]

    X_train = train_data.drop(columns=[target])
    y_train = train_data[target]

    X_test = test_data.drop(columns=[target])
    y_test = test_data[target]

    return X_train,X_test,y_train,y_test

### Performance metrics

In [ ]:
def k_fold_cross_validation(df, target_col, k=5, max_depth=3, min_samples_split=2, bins=3):

    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    fold_size = len(df) // k
    metrics = []

    for i in range(k):
        start, end = i * fold_size, (i + 1) * fold_size
        test_df = df.iloc[start:end]
        train_df = pd.concat([df.iloc[:start], df.iloc[end:]])

        X_train = discretize(train_df.drop(columns=[target_col]), bins=bins).reset_index(drop=True)
        y_train = train_df[target_col].reset_index(drop=True)


        X_test = discretize(test_df.drop(columns=[target_col]), bins=bins).reset_index(drop=True)
        y_test = test_df[target_col].reset_index(drop=True)


        tree = DecisionTreeID3(max_depth=max_depth, min_samples_split=min_samples_split)
        tree.fit(X_train, y_train)
        y_pred = tree.predict(X_test)

        acc = sum(yt == yp for yt, yp in zip(y_test, y_pred)) / len(y_test)
        metrics.append(acc)

    print(f"K-Fold Accuracy por fold: {metrics}")
    print(f"Média: {np.mean(metrics):.4f} | Desvio padrão: {np.std(metrics):.4f}")
    return metrics

def accuracy(predicton,true):
    return sum(yt==yp for yp,yt in zip(predicton,true)) / len(true)

def confusion_matrix(y_true, y_pred, labels=None, title="Confusion Matrix"):
    if labels is None:
        labels = sorted(set(y_true) | set(y_pred))
    
    matrix = pd.DataFrame(
        np.zeros((len(labels), len(labels)), dtype=int),
        index=labels, columns=labels
    )

    for t, p in zip(y_true, y_pred):
        matrix.loc[t, p] += 1

    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=True, 
                xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()

def precision_recall_f1(y_true, y_pred):
    from collections import Counter
    labels = sorted(set(y_true) | set(y_pred))
    tp = Counter()
    fp = Counter()
    fn = Counter()
    for yt, yp in zip(y_true, y_pred):
        if yt == yp:
            tp[yt] += 1
        else:
            fp[yp] += 1
            fn[yt] += 1
    metrics = {}
    for label in labels:
        p = tp[label] / (tp[label] + fp[label]) if (tp[label] + fp[label]) else 0
        r = tp[label] / (tp[label] + fn[label]) if (tp[label] + fn[label]) else 0
        f1 = 2 * p * r / (p + r) if (p + r) else 0
        metrics[label] = {"precision": p, "recall": r, "f1-score": f1}
    return metrics

### ROC curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def one_hot_encode(y, num_classes=None):
    if num_classes is None:
        num_classes = np.max(y) + 1
    return np.eye(num_classes)[y]

def compute_roc_curve(y_true_bin, y_score):
    thresholds = np.linspace(0, 1, num=100)
    tpr_list, fpr_list = [], []
    for thresh in thresholds:
        y_pred = (y_score >= thresh).astype(int)
        TP = np.sum((y_pred == 1) & (y_true_bin == 1))
        FP = np.sum((y_pred == 1) & (y_true_bin == 0))
        FN = np.sum((y_pred == 0) & (y_true_bin == 1))
        TN = np.sum((y_pred == 0) & (y_true_bin == 0))
        TPR = TP / (TP + FN) if TP + FN > 0 else 0
        FPR = FP / (FP + TN) if FP + TN > 0 else 0
        tpr_list.append(TPR)
        fpr_list.append(FPR)
    return np.array(fpr_list), np.array(tpr_list)

def trapezoidal_auc(x, y):

    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]
    return np.trapz(y_sorted, x_sorted)


def plot_roc(y_true, y_proba, class_labels=None):

    y_true = np.array(y_true)
    y_score = np.array(y_proba)
    n_classes = y_score.shape[1]

    
    _, y_true_int = np.unique(y_true, return_inverse=True)
    y_true_bin = np.eye(n_classes)[y_true_int]

    if class_labels is None:
        class_labels = [str(i) for i in range(n_classes)]

    plt.figure(figsize=(8, 6))
    for i in range(n_classes):
        fpr, tpr = compute_roc_curve(y_true_bin[:, i], y_score[:, i])
        auc_val = trapezoidal_auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'Class {class_labels[i]} (AUC = {auc_val:.2f})')

    plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Multiclass ROC Curve')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

### Save the trainded models
dfjfjfj

In [ ]:
def save_model(tree, name='test.pkl'):
    with open(name, 'wb') as f:
        pickle.dump(tree, f)

### Iris training

In [ ]:
def trainIris(kfold=False,split=0.7):
    df=pd.read_csv('iris.csv')

    
    X_train,X_test,y_train,y_test=train_test_split(data=df,split=split)

    tree = DecisionTreeID3(max_depth=3,min_samples_split=3)

    X_train = discretize(X_train)
    X_test = discretize(X_test)


    tree.fit(X_train,y_train)

    y_prediction=tree.predict(X_test)

    y_prob=tree.predict_probability(X_test)
    print("y_prob shape:", y_prob.shape)

    save_model(tree,name='iris_model.pkl')

    print("Accuracy: {:.2f}%".format(accuracy(y_test.tolist(), y_prediction) * 100))

    print("y_prob shape:", np.array(y_prob).shape)

    print("Precision / Recall / F1-score por classe:")
    scores = precision_recall_f1(y_test.tolist(), y_prediction)
    for label, metric in scores.items():
        print(f"    Classe {label}: Precision={metric['precision']:.2f}, Recall={metric['recall']:.2f}, F1={metric['f1-score']:.2f}")
    if kfold:
        k_fold_cross_validation(df,"class")

    confusion_matrix(y_test,y_prediction)
    plot_roc(y_test,y_prob,['setosa' , 'versicolor' , 'virginica'])

In [ ]:
trainIris(split=0.7)

### Connect4 training

In [ ]:
def trainconnect4(max_deph=15,min_split=1,min_gain=1e-2,split=0.7,save=False):
    df=pd.read_csv('mcts_moves.csv')

    X_train,X_test,y_train,y_test=train_test_split(df,split=split,target="move")

    tree= DecisionTreeID3(max_depth=max_deph,min_samples_split=min_split,min_gain=min_gain)

    tree.fit(X_train,y_train)
    if save:
        save_model(tree , "connect4.pkl")

    pred=tree.predict(X_test)

    y_pred_train = tree.predict(X_train)
    acc_train = accuracy(y_pred_train, y_train)
    print(f"Train accuracy: {acc_train:.2f}")

   

    pred_prob=tree.predict_probability(X_test)

    print(accuracy(pred,y_test))

    confusion_matrix(y_test,pred)

    plot_roc(y_test,pred_prob)


### Test hyper parameters

In [ ]:
trainconnect4(max_deph=6,min_split=100,min_gain=1e-2)

In [ ]:
trainconnect4(max_deph=10,min_split=100,min_gain=1e-2)

In [ ]:
trainconnect4(max_deph=15,min_split=100,min_gain=1e-2)

In [ ]:
trainconnect4(max_deph=10,min_split=4,min_gain=1e-4)

In [ ]:
trainconnect4(max_deph=10,min_split=10,min_gain=1e-2)

In [ ]:
trainconnect4(max_deph=10,min_split=1,min_gain=1e-2)

In [ ]:
trainconnect4(max_deph=10,min_split=1,min_gain=1e-20)

In [ ]:
trainconnect4(max_deph=14,min_split=1,min_gain=1e-2)

In [ ]:
trainconnect4(max_deph=15,min_split=1,min_gain=1e-2)

In [ ]:
trainconnect4(max_deph=18,min_split=1,min_gain=1e-2)

In [ ]:
trainconnect4(max_deph=20,min_split=10,min_gain=1e-2)

### Save best model

In [ ]:
trainconnect4(save=True)

### Get decision tree moves

In [ ]:
def ID3(**kwargs):
    board = kwargs.get("board")

    with open('connect4.pkl' , "rb") as f:
        tree = pickle.load(f)
    
    state=board_to_state(board)

    move= tree.predict(pd.DataFrame([state]))[0]

    return move
    
def board_to_state(board):
    state={}
    index= 0
    for row in board:
        for cell in row:
            state[f"s{index}"] = cell
            index+=1
    return state

In [ ]:
play_game(mode='pvc' , ai=ID3)